# Demo de inferencia ventana a ventana

Cuaderno para reproducir el flujo de inferencia casi en tiempo real:
1. Seleccionar el experimento entrenado y una sesión enriched_*.parquet.
2. Ejecutar `run_inference.py`, que aplica el pipeline al vuelo y guarda las predicciones por ventana.
3. Analizar métricas y visualizar `fatigue_pred` frente a `fatigue_score` para documentar el comportamiento del modelo.

### 1. Configuración

Ajusta las rutas según el experimento y la sesión que quieras evaluar.

In [5]:
from pathlib import Path

EXPERIMENT_DIR = Path("../data/results/modeling/experiments/runner_id_20251217_190729")
MODEL_NAME = "gradient_boosting"
ENRICHED_PATH = Path("../data/enriched/enriched_D_231001_runTEST__3_184.0_LPM_2025_10_24_13_06_22_T3_1.parquet")
OUTPUT_PATH = Path("../data/results/modeling/inference/demo_predictions_notebook.parquet")
WINDOW_SECONDS = 3.0
OVERLAP_RATIO = 0.5
PLAYBACK_SPEED = 0.0

EXPERIMENT_DIR, ENRICHED_PATH, OUTPUT_PATH


(PosixPath('../data/results/modeling/experiments/runner_id_20251217_190729'),
 PosixPath('../data/enriched/enriched_D_231001_runTEST__3_184.0_LPM_2025_10_24_13_06_22_T3_1.parquet'),
 PosixPath('../data/results/modeling/inference/demo_predictions_notebook.parquet'))

### 2. Ejecutar run_inference.py

El script reutiliza el pipeline guardado y genera predicciones por ventana.

In [6]:
import subprocess
import shlex

cmd = [
    "python",
    "../src/models/run_inference.py",
    "--enriched", str(ENRICHED_PATH),
    "--experiment", str(EXPERIMENT_DIR),
    "--model", MODEL_NAME,
    "--window", str(WINDOW_SECONDS),
    "--overlap", str(OVERLAP_RATIO),
    "--output", str(OUTPUT_PATH),
    "--playback-speed", str(PLAYBACK_SPEED),
]
print("Comando:", " ".join(shlex.quote(part) for part in cmd))

resultado = subprocess.run(cmd, capture_output=True, text=True)
print("STDOUT:\n", resultado.stdout)
print("STDERR:\n", resultado.stderr)
resultado.check_returncode()


Comando: python ../src/models/run_inference.py --enriched ../data/enriched/enriched_D_231001_runTEST__3_184.0_LPM_2025_10_24_13_06_22_T3_1.parquet --experiment ../data/results/modeling/experiments/runner_id_20251217_190729 --model gradient_boosting --window 3.0 --overlap 0.5 --output ../data/results/modeling/inference/demo_predictions_notebook.parquet --playback-speed 0.0
STDOUT:
 
STDERR:
 2025-12-17 23:47:37,293 - INFO - Evaluation (window-level fatigue_score) -> MAE=0.0210 RMSE=0.0260 R2=0.9624
2025-12-17 23:47:37,296 - INFO - [t=  0.00s] pred=0.664 | score=0.655
2025-12-17 23:47:37,296 - INFO - [t=  1.51s] pred=0.936 | score=0.938
2025-12-17 23:47:37,296 - INFO - [t=  3.01s] pred=0.933 | score=0.928
2025-12-17 23:47:37,296 - INFO - [t=  4.50s] pred=0.662 | score=0.646
2025-12-17 23:47:37,297 - INFO - [t=  6.00s] pred=0.957 | score=0.961
2025-12-17 23:47:37,297 - INFO - [t=  7.50s] pred=0.935 | score=0.961
2025-12-17 23:47:37,297 - INFO - [t=  9.02s] pred=0.726 | score=0.703
2025-12

### 3. Cargar predicciones y revisar métricas

En este paso verificamos el rendimiento del modelo y comprobamos que la inferencia sea consistente con los resultados de entrenamiento.

In [7]:
import pandas as pd
import numpy as np

pred_df = pd.read_parquet(OUTPUT_PATH)
pred_df.head()

,file,source_file,start_s,duration,n_samples,acc_x_centered_mean,acc_x_centered_std,acc_x_centered_mad,acc_x_centered_skew,acc_x_centered_kurt,...,grav_z_skew,grav_z_kurt,jerk_mean,jerk_std,jerk_mad,jerk_skew,hr_mean,spo2_mean,fatigue_score,fatigue_pred
0,clean_D_231001_runTEST__3_184.0_LPM_2025_10_24...,enriched_D_231001_runTEST__3_184.0_LPM_2025_10...,0.000000,2.996452,301,0.029476,0.811144,0.556568,-0.120877,-1.303989,...,-0.447481,-0.820043,32.156627,24.086299,15.790979,1.216937,177.0,99.0,0.655,0.664105
1,clean_D_231001_runTEST__3_184.0_LPM_2025_10_24...,enriched_D_231001_runTEST__3_184.0_LPM_2025_10...,1.508280,2.986378,300,0.037904,0.766258,0.578933,-0.219545,-1.482254,...,0.019903,-0.990459,91.730517,398.834136,14.166484,6.692005,177.0,99.0,0.938,0.935895
2,clean_D_231001_runTEST__3_184.0_LPM_2025_10_24...,enriched_D_231001_runTEST__3_184.0_LPM_2025_10...,3.006426,2.986345,300,0.046608,0.803415,0.576618,-0.297721,-1.488704,...,0.210968,-0.035865,91.811495,398.899950,13.179422,6.688007,177.0,99.0,0.928,0.932717
3,clean_D_231001_runTEST__3_184.0_LPM_2025_10_24...,enriched_D_231001_runTEST__3_184.0_LPM_2025_10...,4.504585,2.986360,300,0.029927,0.827674,0.618432,-0.271914,-1.470773,...,-0.689947,1.954540,30.425191,23.484279,12.509150,1.452029,177.0,99.0,0.646,0.662350
4,clean_D_231001_runTEST__3_184.0_LPM_2025_10_24...,enriched_D_231001_runTEST__3_184.0_LPM_2025_10...,6.002839,2.986050,300,-0.015046,0.805382,0.699752,-0.080054,-1.457823,...,-0.827550,1.630825,55.549393,306.707395,13.758911,11.963302,177.0,99.0,0.961,0.956552


In [8]:
print(f"Ventanas totales: {len(pred_df):,}")

Ventanas totales: 58


In [9]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

metrics = {}
if "fatigue_score" in pred_df.columns and not pred_df["fatigue_score"].isna().all():
    y_true = pred_df["fatigue_score"].to_numpy()
    y_pred = pred_df["fatigue_pred"].to_numpy()
    metrics = {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred, squared=False),
        "R2": r2_score(y_true, y_pred),
    }

{clave: round(valor, 4) for clave, valor in metrics.items()}


{'MAE': 0.021, 'RMSE': 0.026, 'R2': 0.9624}

### 4. Visualizaciones

Comparación temporal y análisis de dispersión entre `fatigue_pred` y `fatigue_score`.

In [10]:
import plotly.express as px

if "fatigue_score" in pred_df.columns and not pred_df["fatigue_score"].isna().all():
    long_df = pred_df.melt(
        id_vars=["start_s"],
        value_vars=["fatigue_pred", "fatigue_score"],
        var_name="serie",
        value_name="valor",
    )
else:
    long_df = pred_df[["start_s", "fatigue_pred"]].assign(serie="fatigue_pred", valor=pred_df["fatigue_pred"])

fig = px.line(
    long_df,
    x="start_s",
    y="valor",
    color="serie",
    title="Predicción vs. valor real por ventana",
)
fig.update_layout(xaxis_title="Tiempo (s)", yaxis_title="fatigue_score")
fig.show()



In [11]:
if "fatigue_score" in pred_df.columns and not pred_df["fatigue_score"].isna().all():
    fig = px.scatter(
        pred_df,
        x="fatigue_score",
        y="fatigue_pred",
        title="Dispersión score real vs. predicho",
        labels={"fatigue_score": "Score real", "fatigue_pred": "Score predicho"},
    )
    min_val = pred_df["fatigue_score"].min()
    max_val = pred_df["fatigue_score"].max()
    fig.add_shape(
        type="line",
        x0=min_val,
        x1=max_val,
        y0=min_val,
        y1=max_val,
        line=dict(color="gray", dash="dash"),
    )
    fig.update_layout(xaxis_title="Score real", yaxis_title="Score predicho")
    fig.show()
else:
    print("No hay score real en el archivo; solo se muestra la serie predicha.")